##### Copyright 2025 Perceptron AI.

In [ ]:
# Licensed under the MIT License (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# https://opensource.org/licenses/MIT
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Capability — PPE Detection (Isaac 0.3 Max)
Locate protective equipment on an assembly line and highlight every instance with normalized Perceptron geometry.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/perceptron-ai-inc/perceptron/blob/main/cookbook/recipes/capabilities/isaac-0.3-max/object-detection.ipynb)

## Install dependencies

In [ ]:
%pip install --upgrade perceptron --quiet

## Download assets

In [ ]:
IMAGE_URL = "https://raw.githubusercontent.com/perceptron-ai-inc/perceptron/main/cookbook/_shared/assets/capabilities/detection/ppe_line.webp"

!curl -so ppe_line.webp {IMAGE_URL}

## Configure the Perceptron client
Authenticate once and point the SDK at Isaac 0.3 Max.

In [ ]:
import os
from pathlib import Path

from IPython.display import Image as IPyImage
from IPython.display import display
from PIL import Image, ImageDraw

from perceptron import configure, image, perceive, text

api_key = os.getenv("PERCEPTRON_API_KEY", "<your Perceptron API key>")
if not api_key or api_key.startswith("<"):
    raise RuntimeError("Set PERCEPTRON_API_KEY or replace the placeholder in this cell.")

configure(
    provider="perceptron",
    model="isaac-0.3-max",
    api_key=api_key,
)

SCENE_PATH = "ppe_line.webp"
ANNOTATED_PATH = Path("ppe_line_annotated.png")

## Build a detection helper
Use the `@perceive` decorator with `expects="box"` so each detection returns normalized bounding boxes.

In [ ]:
TARGET_CLASSES = ["safety helmet", "safety vest"]


@perceive(expects="box", allow_multiple=True)
def detect_ppe(frame_path):
    frame = image(frame_path)
    classes_text = ", ".join(TARGET_CLASSES)
    prompt = text(
        "Find every worker wearing PPE. Focus on helmets and high-visibility vests. "
        "Return one bounding box per instance and include the item name in the mention attribute."
    )
    return frame + prompt

## Run the detection request
Invoke the helper on the PPE line image to retrieve grounded regions.

In [ ]:
detection = detect_ppe(str(SCENE_PATH))
print(detection.text)
boxes = detection.boxes or []
print(f"Returned {len(boxes)} boxes")

## Render grounded results
Convert the normalized coordinates to pixels and overlay them for quick inspection.

In [ ]:
img = Image.open(SCENE_PATH).convert("RGB")
draw = ImageDraw.Draw(img)


def to_px(point):
    return point.x / 1000 * img.width, point.y / 1000 * img.height


for box in boxes:
    top_left = to_px(box.top_left)
    bottom_right = to_px(box.bottom_right)
    draw.rectangle([top_left, bottom_right], outline="dodgerblue", width=3)
    draw.text(top_left, box.mention or "ppe", fill="dodgerblue")

img.save(ANNOTATED_PATH)
display(IPyImage(filename=str(ANNOTATED_PATH)))
print(f"Saved annotated output to {ANNOTATED_PATH}")

## Conclusion & next steps
- Adjust `TARGET_CLASSES` and the prompt to fit your environment.
- Enable `stream=True` inside `@perceive` for incremental detections.
- Add exemplar shots (see the [in-context learning recipe](https://github.com/perceptron-ai-inc/perceptron/blob/main/cookbook/recipes/capabilities/isaac-0.3-max/in-context-learning-image.ipynb)) when classes are ambiguous.
- Pass `reasoning=True` for harder detection tasks.